# Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

• Tracking agent behavior with logging, analytics, and debugging.

• Transforming prompts, tool selection, and output formatting.

• Adding retries, fallbacks, and early termination logic.

• Applying rate limits, guardrails, and PIl detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")  



### Summarization Middleware
Automatically Sumarize conversastion History when approaching tokens limit

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

agent = create_agent(
    model="gpt-4o",
    checkpointer=MemorySaver(),                     
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            summary_length=200,
            trigger=("messages",10),  
            keep_last_n_messages=3
        )
    ],
    system_prompt="You are a helpful assistant.",   
)

### Run with a thread id

In [14]:
config={"configurable":{"thread_id":"test_thread_1"}}


In [7]:
questions = [
    "What is the capital of France?",
    "What is the population of France?",
    "What is the currency of France?",
    "What is the official language of France?",
    "What is the largest city in France?",
    "What is the climate like in France?",
    "What are some popular tourist attractions in France?",
    "What is the history of France?",
    "What is the government structure of France?"
]

In [15]:
for question in questions:
    response = agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=config
    )
    print(f"Q: {question}")
    print(f"A: {response['messages'][-1].content}\n")

Q: What is the capital of France?
A: The capital of France is Paris.

Q: What is the population of France?
A: As of the most recent estimates in 2023, France has a population of approximately 68 million people. Keep in mind that population figures are continually updated as new data becomes available.

Q: What is the currency of France?
A: The currency of France is the euro, which is symbolized as € and abbreviated as EUR. France adopted the euro in 1999 for electronic transactions and introduced euro banknotes and coins in 2002, replacing the French franc.

Q: What is the official language of France?
A: The official language of France is French.

Q: What is the largest city in France?
A: The largest city in France is Paris. It is not only the capital city but also the most populous city in the country.

Q: What is the climate like in France?
A: France has a diverse climate due to its geographical size and varied topography. The climate can generally be categorized into a few types:

1

### Human In The Loop

• High-stakes operations requiring human approval (e.g. database writes, financial transactions).

• Compliance workflows where human oversight is mandatory.

• Long-running conversations where human feedback guides the agent.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [10]:
agent = create_agent(
    model="gpt-4o",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject", "edit"],
                },
                "read_email_tool": False
            }
        )
    ],
    system_prompt="You are an email assistant. You can read and send emails based on user requests."
)

In [11]:
config = {"configurable": {"thread_id": "test-approve"}}
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'Hi John, how are you?'")]},
    config=config
)
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'Hi John, how are you?'", additional_kwargs={}, response_metadata={}, id='c9641cc8-b754-42c5-a3c0-f038abc99a8f'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 117, 'total_tokens': 148, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_ffd8308b42', 'id': 'chatcmpl-EKkf4HUTbMmU9sady2X3FO1lSihgd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a071bd-6937-7980-923c-2e39b9a6744d-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'Hi John, how are you?'}, 'id': 'ca

In [15]:
# step 2
from langgraph.types import Command


if "__interrupt__" in result:
    print("paused for approval")
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

print(f"Result: {result['messages'][-1].content}")

paused for approval
Result: The email has been sent to john@test.com with the subject "Hello."
